
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [5]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002186336A630>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021863369280>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int=Field(description="movie released year")
    director:str=Field(description="director of the movie")
    rating:float=Field(description="movie rating out of 10")


In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002186336A630>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021863369280>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'pa

In [8]:
model_with_structure.invoke("amazing spiderman movie")

Movie(title='Spider-Man: Into the Spider-Verse', year=2018, director='Peter Ramsey, Rodney Rothman, Bob Persichetti', rating=8.4)

### MEssage output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(...,description="the title of the movie")
    year:int=Field(...,description="movie released year")
    director:str=Field(...,description="director of the movie")
    rating:float=Field(...,description="movie rating out of 10")

output = model.with_structured_output(Movie, include_raw=True)
output.invoke("leo movie tamil?")


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user: "leo ... movie tamil?" Likely asking about the Tamil movie "Leo". Could be asking for details. We can provide info about the Tamil movie "Leo" starring Vijay, directed by Lokesh Kanagaraj, etc. Provide rating, year, etc. Could also use function Movie to return structured data. The function expects director, rating, title, year. We can call function with those details.\n\nWe need to provide answer. Use function. Let\'s call with title "Leo", director "Lokesh Kanagaraj", rating maybe 7.5? Not sure. Could approximate. Provide year 2023. Let\'s call function.', 'tool_calls': [{'id': 'fc_c2144c7e-72a9-4e00-8cd0-7e7570954fce', 'function': {'arguments': '{"director":"Lokesh Kanagaraj","rating":7.5,"title":"Leo","year":2023}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 187, 'prompt_tokens': 150, 'total_tokens': 337, 'completion_time': 0.392317026, 'completion_t

### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None, description="budget in millions usd")

model_with_structure = model.with_structured_output(MovieDetails)
model_with_structure.invoke("Provide details about the movie Inception")


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [15]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    ttile:Annotated[str,...,"title of the movie"]
    year:Annotated[int,...,"movie released year"]
    director:Annotated[str,...,"director of the movie"]
    rating:Annotated[float,...,"rating out of 10"]

model_typed_dict = model.with_structured_output(MovieDict)
model_typed_dict.invoke("the avengers")

{'director': 'Joss Whedon', 'rating': 8, 'ttile': 'The Avengers', 'year': 2012}

In [16]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [17]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [19]:
from langchain.agents import create_agent

In [20]:
class ContactInfo(BaseModel):
    name:str=Field(description="name of the person")
    email:str=Field(description="email of that person")
    phone:str=Field(description="phone number of the person")

agent = create_agent(model="groq:openai/gpt-oss-120b", response_format=ContactInfo)
result = agent.invoke({"messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]})
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='b42b58eb-de0a-45f8-b041-3485d7441c8d'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from given text. The response format is defined: ContactInfo JSON schema with fields name, email, phone required. Must output only final JSON object, compact.\n\nWe need to output:\n\n{\n "name":"John Doe",\n "email":"john@example.com",\n "phone":"(555) 123-4567"\n}\n\nMake sure compact (no spaces?). "compact JSON formatting" means no unnecessary whitespace, but typical minimal spaces after commas maybe okay. Let\'s output without spaces: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}\n\nCheck JSON valid. Yes.\n\n'}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 226

In [24]:
## Typedict

from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [23]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')